# Step 1 - Filter the raw Lichess file down to the study games

**Goal:** stream through the full monthly Lichess file once, apply the *game-level* inclusion criteria one at a
time, and save the surviving games (with their Lichess engine evaluations) as a small PGN file.

**Input:** `data/raw/lichess_db_standard_rated_2026-10.pgn.zst` (the file your downloader fetched; pinned month in `config.py`).

**Output** (in `outputs/step1_filter_games/`):
- `games_filtered.pgn` - the games that passed every criterion (input of step 2)
- `funnel.csv` - how many games remain after each criterion (for the thesis funnel table / diagram)
- `manifest.json` - config, package versions, SHA-256 of the raw file and of the output = reproducibility record

**Funnel (each criterion is its own step, as your professor asked):**
both players >= 2600 -> rating gap <= 100 -> rapid or slower -> normal termination -> no bot accounts ->
at least 60 plies -> has Lichess evaluations.

The pass is deterministic (no randomness), so the same input file always gives the same output file.
It reads the whole file, so expect a long run; use `MAX_GAMES` for a quick test first.

In [ ]:
import io, re, json, sys
from pathlib import Path
import pandas as pd
import config as C
from common import sha256_file, write_manifest

RAW = C.RAW_DIR / C.RAW_FILE
OUT = C.OUT_DIR / "step1_filter_games"
OUT.mkdir(parents=True, exist_ok=True)
MAX_GAMES = None        # e.g. 2_000_000 for a quick test run; None = whole file
print("Input:", RAW, "| exists:", RAW.exists())

## Optional: download the file (skip if you already have it)
Streams to disk, so it needs free space (a monthly file is tens of GB). Leave `DOWNLOAD = False` if you use your own downloader.

In [ ]:
DOWNLOAD = False
if DOWNLOAD and not RAW.exists():
    import requests
    RAW.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(C.LICHESS_URL, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(RAW, "wb") as f:
            for chunk in r.iter_content(1 << 22):
                f.write(chunk)

## The criteria
`first_fail` returns the *first* criterion a game fails (or `None` if it passes all). Cheap header checks come first,
the expensive movetext checks (number of plies, evaluations) last.
- **Rapid or slower:** `base + 40 * increment >= 480` seconds (Lichess' own rule); games without a clock are dropped.
- **Normal termination:** the `Termination` tag is `Normal` (no abandoned / time-forfeit games).
- **No bots:** neither player has the `BOT` title.

In [ ]:
ORDER = ["elo", "gap", "rapid_or_slower", "normal_termination", "no_bot", "min_plies", "has_eval"]
LABELS = {"elo": f"both players >= {C.ELO_MIN}", "gap": f"rating gap <= {C.MAX_ELO_GAP}",
          "rapid_or_slower": "rapid or slower", "normal_termination": "normal termination",
          "no_bot": "no bot accounts", "min_plies": f">= {C.MIN_PLIES} plies", "has_eval": "has Lichess evaluations"}
TAG = re.compile(r'^\[(\w+) "(.*)"\]$')
COMMENT = re.compile(r"\{[^}]*\}")
MOVENUM = re.compile(r"\d+\.(\.\.)?")
RESULTS = {"1-0", "0-1", "1/2-1/2", "*"}

def est_seconds(tc):
    m = re.fullmatch(r"(\d+)\+(\d+)", tc or "")
    return int(m[1]) + 40 * int(m[2]) if m else None

def n_plies(movetext):
    toks = COMMENT.sub(" ", movetext).split()
    return sum(1 for t in toks if not MOVENUM.fullmatch(t) and t not in RESULTS and not t.startswith("$"))

def first_fail(h, mt):
    try:
        w, b = int(h["WhiteElo"]), int(h["BlackElo"])
    except (KeyError, ValueError):
        return "elo"
    if min(w, b) < C.ELO_MIN: return "elo"
    if abs(w - b) > C.MAX_ELO_GAP: return "gap"
    est = est_seconds(h.get("TimeControl"))
    if est is None or est < C.MIN_EST_SECONDS: return "rapid_or_slower"
    if h.get("Termination") != "Normal": return "normal_termination"
    if "BOT" in (h.get("WhiteTitle"), h.get("BlackTitle")): return "no_bot"
    if n_plies(mt) < C.MIN_PLIES: return "min_plies"
    if "[%eval" not in mt: return "has_eval"
    return None

def open_text(path):
    if str(path).endswith(".zst"):
        import zstandard
        raw = open(path, "rb")
        return io.TextIOWrapper(zstandard.ZstdDecompressor(max_window_size=2**31).stream_reader(raw),
                                encoding="utf-8", errors="replace")
    return open(path, encoding="utf-8", errors="replace")

## The streaming pass
Lichess PGN files have one movetext line per game. Games that pass are written out immediately.

In [ ]:
fails = {k: 0 for k in ORDER}
total, kept = 0, 0
out_pgn = OUT / "games_filtered.pgn"
with open_text(RAW) as fh, open(out_pgn, "w", encoding="utf-8", newline="\n") as fo:
    h, hl = {}, []
    for line in fh:
        line = line.rstrip("\n")
        if line.startswith("["):
            m = TAG.match(line)
            if m:
                h[m[1]] = m[2]; hl.append(line)
        elif line.strip():
            total += 1
            f = first_fail(h, line)
            if f is None:
                kept += 1
                fo.write("\n".join(hl) + "\n\n" + line + "\n\n")
            else:
                fails[f] += 1
            h, hl = {}, []
            if total % 1_000_000 == 0:
                print(f"{total:,} games read, {kept} kept", flush=True)
            if MAX_GAMES and total >= MAX_GAMES:
                break
print(f"Done: {total:,} games read, {kept} kept -> {out_pgn}")

## Funnel table
`games_remaining` = games that passed this criterion *and all before it*.

In [ ]:
rows, remaining = [dict(step="all games in file", games_remaining=total, dropped_here=0)], total
for k in ORDER:
    remaining -= fails[k]
    rows.append(dict(step=LABELS[k], games_remaining=remaining, dropped_here=fails[k]))
funnel = pd.DataFrame(rows)
funnel.to_csv(OUT / "funnel.csv", index=False)
assert remaining == kept
funnel

## Reproducibility record
The SHA-256 of the raw file ties the result to the exact input (it is slow for a very large file, but only needs to run once;
set `HASH_RAW = False` to skip during test runs).

In [ ]:
HASH_RAW = True
write_manifest(OUT / "manifest.json", "step1_filter_games",
               {"raw_file": RAW.name, "raw_sha256": sha256_file(RAW) if HASH_RAW else None,
                "max_games": MAX_GAMES, "games_read": total, "games_kept": kept,
                "output_sha256": sha256_file(out_pgn), "funnel": rows})
print("Manifest written.")